# Initial Inspection — Engelen-cleaned CIC-IDS2017

**Notebook 01 of EDA series**
**Author:** Bradly Morein (UP2088701)
**Date:** 26 April 2026

## Purpose

First pass over the Engelen-corrected CIC-IDS2017 Parquet files. Goals:

1. Confirm the data loads cleanly from Parquet.
2. Document column names, dtypes, and basic statistics.
3. Examine the label column and per-day class distributions.
4. Identify questions worth investigating in more depth.

This notebook is exploratory only. Production preprocessing logic lives in
`src/preprocessing/`, not here.

## Data source

- **Dataset:** Engelen et al. (2021) corrected CIC-IDS2017
- **Location:** `data/interim/cicids2017-engelen/*.parquet`
- **SHA-256 record:** `data/sha256sums.txt`
- **ADR for dataset choice:** `docs/decisions/0001-primary-dataset.md`

In [1]:
"""Imports and notebook-wide settings."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repository root, regardless of where the notebook is opened
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PARQUET_DIR = REPO_ROOT / "data" / "interim" / "cicids2017-engelen"

# Display settings — show more columns/rows in pandas output
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 50)

print(f"Repo root: {REPO_ROOT}")
print(f"Parquet directory: {PARQUET_DIR}")
print(f"Files present: {sorted(p.name for p in PARQUET_DIR.glob('*.parquet'))}")

Repo root: /home/brad/projects/dissertation
Parquet directory: /home/brad/projects/dissertation/data/interim/cicids2017-engelen
Files present: ['Friday-WorkingHours.parquet', 'Monday-WorkingHours.parquet', 'Thursday-WorkingHours.parquet', 'Tuesday-WorkingHours.parquet', 'Wednesday-WorkingHours.parquet']


In [2]:
"""Load Monday's data — the benign-only baseline day — and inspect."""
df_mon = pd.read_parquet(PARQUET_DIR / "Monday-WorkingHours.parquet")

print(f"Shape: {df_mon.shape[0]:,} rows × {df_mon.shape[1]} columns")
print(f"Memory footprint: {df_mon.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
print(f"\nFirst 3 rows of the first 8 columns:")
df_mon.iloc[:3, :8]

Shape: 371,749 rows × 84 columns
Memory footprint: 293 MB

First 3 rows of the first 8 columns:


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration
0,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,03/07/2017 01:56:34 PM,119719148
1,192.168.10.9-192.168.10.3-123-123-17,192.168.10.9,123,192.168.10.3,123,17,03/07/2017 01:56:55 PM,65511209
2,192.168.10.12-224.0.0.251-5353-5353-17,192.168.10.12,5353,224.0.0.251,5353,17,03/07/2017 01:57:21 PM,113976922


In [3]:
"""Audit columns and dtypes — what is CICFlowMeter actually giving us?"""
print(f"Total columns: {len(df_mon.columns)}\n")

dtype_counts = df_mon.dtypes.value_counts()
print("Columns by dtype:")
print(dtype_counts)

print("\nObject (string) columns:")
for col in df_mon.select_dtypes(include="object").columns:
    print(f"  {col}")

print("\nFirst 20 numeric columns:")
for col in df_mon.select_dtypes(include="number").columns[:20]:
    print(f"  {col}")

Total columns: 84

Columns by dtype:
float32    45
int64      34
object      5
Name: count, dtype: int64

Object (string) columns:
  Flow ID
  Src IP
  Dst IP
  Timestamp
  Label

First 20 numeric columns:
  Src Port
  Dst Port
  Protocol
  Flow Duration
  Total Fwd Packet
  Total Bwd packets
  Total Length of Fwd Packet
  Total Length of Bwd Packet
  Fwd Packet Length Max
  Fwd Packet Length Min
  Fwd Packet Length Mean
  Fwd Packet Length Std
  Bwd Packet Length Max
  Bwd Packet Length Min
  Bwd Packet Length Mean
  Bwd Packet Length Std
  Flow Bytes/s
  Flow Packets/s
  Flow IAT Mean
  Flow IAT Std


In [4]:
"""Examine the Label column — the most important column in the dataset."""
label_cols = [c for c in df_mon.columns if "label" in c.lower()]
print(f"Label-like columns found: {label_cols}")

label_col = label_cols[0]
print(f"Using label column: '{label_col}'")
print(f"Dtype: {df_mon[label_col].dtype}")
print(f"\nUnique values and counts:")
print(df_mon[label_col].value_counts(dropna=False))

print(f"\nNaN count: {df_mon[label_col].isna().sum()}")
print(f"Total rows: {len(df_mon)}")

Label-like columns found: ['Label']
Using label column: 'Label'
Dtype: object

Unique values and counts:
Label
BENIGN    371749
Name: count, dtype: int64

NaN count: 0
Total rows: 371749


## Observations from Monday inspection

1. **Schema:** 84 columns, 5 string + 79 numeric (45 float32, 34 int64).
2. **String columns:** `Flow ID`, `Src IP`, `Dst IP`, `Timestamp`, `Label` — first
   four to be dropped or set aside before training; `Label` to be encoded.
3. **Column naming inconsistencies inherited from CICFlowMeter** (e.g.,
   `Total Fwd Packet` vs `Total Bwd packets`). Code must match exact names.
4. **Date format** is DD/MM/YYYY (UK/European), not US format. Parse accordingly.
5. **Capture network is `192.168.10.0/24`** — RFC 1918 private space, lab testbed.
6. **Label column** is `Label` (no leading space — Engelen has fixed the
   classic CICFlowMeter naming bug).
7. **Monday is benign-only:** 371,749 rows, all `BENIGN`, no NaN labels.
   Suitable as the AE training set.
8. **Pre-processing decisions implied:**
   - Drop `Flow ID`, `Src IP`, `Dst IP` before training (network topology leak).
   - Set `Timestamp` aside from training features but retain for chronological splits.
   - Filter rows where Protocol = 0 (non-IP frames recorded as artifacts) — count to be measured in next cell.

## Contrast: Wednesday — the heaviest attack day

Wednesday hosted multiple DoS attacks and the Heartbleed vulnerability exploit.
Comparing its label distribution to Monday's tells us:
- The class imbalance ratio we'll be working with.
- Which attack categories are present in CICIDS2017.
- How clean the labels remain after Engelen's cleaning.

In [5]:
"""Load Wednesday and compare label distribution."""
df_wed = pd.read_parquet(PARQUET_DIR / "Wednesday-WorkingHours.parquet")

print(f"Shape: {df_wed.shape[0]:,} rows × {df_wed.shape[1]} columns\n")

print(f"Label distribution:")
wed_labels = df_wed["Label"].value_counts(dropna=False)
print(wed_labels)

# Compute the imbalance — what fraction is benign vs attack
benign_count = wed_labels.get("BENIGN", 0)
attack_count = len(df_wed) - benign_count
print(f"\nBenign: {benign_count:,} ({100 * benign_count / len(df_wed):.2f}%)")
print(f"Attack: {attack_count:,} ({100 * attack_count / len(df_wed):.2f}%)")
print(f"Imbalance ratio (benign:attack): {benign_count / attack_count:.1f}:1" 
      if attack_count > 0 else "All benign")

# Schema sanity check — do Monday and Wednesday have identical columns?
mon_cols = set(df_mon.columns)
wed_cols = set(df_wed.columns)
if mon_cols == wed_cols:
    print(f"\nSchema check: PASS — Monday and Wednesday have identical columns.")
else:
    print(f"\nSchema check: FAIL")
    print(f"  In Monday but not Wednesday: {mon_cols - wed_cols}")
    print(f"  In Wednesday but not Monday: {wed_cols - mon_cols}")

Shape: 496,779 rows × 84 columns

Label distribution:
Label
BENIGN                          319216
DoS Hulk                        158469
DoS GoldenEye                     7567
DoS slowloris                     4001
DoS Slowhttptest - Attempted      3369
DoS Slowhttptest                  1742
DoS slowloris - Attempted         1731
DoS Hulk - Attempted               593
DoS GoldenEye - Attempted           80
Heartbleed                          11
Name: count, dtype: int64

Benign: 319,216 (64.26%)
Attack: 177,563 (35.74%)
Imbalance ratio (benign:attack): 1.8:1

Schema check: PASS — Monday and Wednesday have identical columns.


## Observations from Wednesday inspection

1. **Schema is consistent** across Monday (benign-only) and Wednesday
   (attack-heavy). All five days expected to share the 84-column structure.
2. **Engelen has split each DoS attack into "successful" and "Attempted"
   variants** — this granularity is *not* present in the original CICIDS2017
   release. It's a methodological feature of the corrected dataset, not a
   data quality problem.
3. **Decision required:** how to handle "Attempted" labels in encoding.
   Plan: Option A (binary, *-Attempted treated as attack) for primary results,
   Option B (Attempted treated as benign) as a sensitivity analysis. Document
   in Methodology chapter §4.3.
4. **Heartbleed is a micro-class** (11 flows on Wednesday). Per-class
   analysis in Chapter 6 must report Heartbleed separately, not just
   macro-averaged.
5. **Wednesday's 1.8:1 imbalance is artificially low** because of DoS Hulk's
   high flow volume. The whole-dataset ratio (next cell) will be the real
   imbalance to design around.

## Observations from Wednesday inspection

1. **Schema is consistent** across Monday (benign-only) and Wednesday
   (attack-heavy). All five days expected to share the 84-column structure.
2. **Engelen has split each DoS attack into "successful" and "Attempted"
   variants** — this granularity is *not* present in the original CICIDS2017
   release. It's a methodological feature of the corrected dataset, not a
   data quality problem.
3. **Decision required:** how to handle "Attempted" labels in encoding.
   Plan: Option A (binary, *-Attempted treated as attack) for primary results,
   Option B (Attempted treated as benign) as a sensitivity analysis. Document
   in Methodology chapter §4.3.
4. **Heartbleed is a micro-class** (11 flows on Wednesday). Per-class
   analysis in Chapter 6 must report Heartbleed separately, not just
   macro-averaged.
5. **Wednesday's 1.8:1 imbalance is artificially low** because of DoS Hulk's
   high flow volume. The whole-dataset ratio (next cell) will be the real
   imbalance to design around.

## Whole-dataset class distribution

Aggregating across all five days. This is the imbalance ratio that drives
preprocessing decisions (class weighting, SMOTE strategy, threshold tuning).

In [6]:
"""Aggregate label distribution across all five days."""
day_files = sorted(PARQUET_DIR.glob("*.parquet"))
print(f"Aggregating across {len(day_files)} days:\n")

all_labels = pd.Series(dtype="object")
day_summary = []

for path in day_files:
    df = pd.read_parquet(path, columns=["Label"])
    day = path.stem.replace("-WorkingHours", "")
    n_total = len(df)
    n_benign = (df["Label"] == "BENIGN").sum()
    n_attack = n_total - n_benign
    day_summary.append({
        "day": day,
        "total": n_total,
        "benign": n_benign,
        "attack": n_attack,
        "attack_pct": round(100 * n_attack / n_total, 2),
    })
    all_labels = pd.concat([all_labels, df["Label"]], ignore_index=True)

print("Per-day summary:")
summary_df = pd.DataFrame(day_summary)
print(summary_df.to_string(index=False))

print(f"\nWhole-dataset label distribution:")
total_labels = all_labels.value_counts(dropna=False)
print(total_labels.head(20))
print(f"  ... and {max(0, len(total_labels) - 20)} more labels not shown")

n_total_all = len(all_labels)
n_benign_all = (all_labels == "BENIGN").sum()
n_attack_all = n_total_all - n_benign_all

print(f"\nTotal flows: {n_total_all:,}")
print(f"Benign: {n_benign_all:,} ({100 * n_benign_all / n_total_all:.2f}%)")
print(f"Attack: {n_attack_all:,} ({100 * n_attack_all / n_total_all:.2f}%)")
print(f"Imbalance ratio (benign:attack): {n_benign_all / n_attack_all:.1f}:1")

print(f"\nUnique label values across whole dataset: {len(total_labels)}")

Aggregating across 5 days:

Per-day summary:
      day  total  benign  attack  attack_pct
   Friday 547915  291433  256482       46.81
   Monday 371749  371749       0        0.00
 Thursday 362368  360264    2104        0.58
  Tuesday 322003  315031    6972        2.17
Wednesday 496779  319216  177563       35.74

Whole-dataset label distribution:
BENIGN                                  1657693
PortScan                                 159151
DoS Hulk                                 158469
DDoS                                      95123
DoS GoldenEye                              7567
DoS slowloris                              4001
FTP-Patator                                3973
DoS Slowhttptest - Attempted               3369
SSH-Patator                                2980
DoS Slowhttptest                           1742
DoS slowloris - Attempted                  1731
Bot - Attempted                            1470
Web Attack - Brute Force - Attempted       1214
Bot                       

In [7]:
"""See all 25 unique labels — including the rare ones cut off above."""
print(f"All {len(total_labels)} unique labels in CIC-IDS2017 (Engelen-cleaned):\n")
for label, count in total_labels.items():
    pct = 100 * count / n_total_all
    flag = "  micro-class" if count < 100 else ""
    print(f"  {count:>10,}  ({pct:>5.2f}%)  {label}{flag}")

All 25 unique labels in CIC-IDS2017 (Engelen-cleaned):

   1,657,693  (78.91%)  BENIGN
     159,151  ( 7.58%)  PortScan
     158,469  ( 7.54%)  DoS Hulk
      95,123  ( 4.53%)  DDoS
       7,567  ( 0.36%)  DoS GoldenEye
       4,001  ( 0.19%)  DoS slowloris
       3,973  ( 0.19%)  FTP-Patator
       3,369  ( 0.16%)  DoS Slowhttptest - Attempted
       2,980  ( 0.14%)  SSH-Patator
       1,742  ( 0.08%)  DoS Slowhttptest
       1,731  ( 0.08%)  DoS slowloris - Attempted
       1,470  ( 0.07%)  Bot - Attempted
       1,214  ( 0.06%)  Web Attack - Brute Force - Attempted
         738  ( 0.04%)  Bot
         652  ( 0.03%)  Web Attack - XSS - Attempted
         593  ( 0.03%)  DoS Hulk - Attempted
         151  ( 0.01%)  Web Attack - Brute Force
          80  ( 0.00%)  DoS GoldenEye - Attempted  micro-class
          32  ( 0.00%)  Infiltration  micro-class
          27  ( 0.00%)  Web Attack - XSS  micro-class
          16  ( 0.00%)  Infiltration - Attempted  micro-class
          12  ( 0.00%

## Observations from full label distribution

**Class spectrum: 207,000:1 between most and least common class.** 
BENIGN (1,657,693) to SSH-Patator-Attempted (8). PR-AUC justification 
empirically grounded; macro-averaged metrics will be uninformative.

**8 micro-classes with under 100 flows each:**
- DoS GoldenEye-Attempted (80), Infiltration (32), Web Attack-XSS (27),
- Infiltration-Attempted (16), Web Attack-Sql Injection (12), 
- FTP-Patator-Attempted (11), Heartbleed (11), SSH-Patator-Attempted (8).

**Design decisions implied:**
1. Primary modelling task is **binary classification** (BENIGN vs any attack).
   Aggregates all attack labels into one class of 443,121 — eliminates the
   micro-class problem for the headline experiment.
2. Per-attack analysis in Chapter 6 will report micro-classes separately,
   with explicit acknowledgement of wide confidence intervals on tiny
   test sets.
3. Heartbleed (11 instances total) is a well-known limitation of CICIDS2017;
   cite as such in §7.4 (Limitations).

**Observation worth working into the Discussion chapter:** Engelen's attempt
vs successful split reveals divergent attack-success ratios. Slow-pace web 
attacks (Slowhttptest, XSS) show more attempted than successful flows, 
consistent with their operational profile as easily-blocked HTTP request 
floods. Brute-force tools (FTP/SSH-Patator) appear predominantly as 
"successful" flows, likely reflecting labelling convention rather than 
attack outcome.

**Capitalisation inconsistency in CIC labels:** `XSS` uppercase, 
`Sql Injection` title case. Document any normalisation applied during 
encoding.